In [2]:
# %% [markdown]
# # 09 — Fraud / Abuse Rules and Anomalies
#
# ## Objective
# Build the first fraud / abuse suspicion framework at:
# - member level
# - provider level
#
# Main outputs:
# - `member_abuse_flags.csv`
# - `provider_fraud_flags.csv`

# %%
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

# %%
import pandas as pd
import numpy as np

from src.config import PROCESSED_DIR, OUTPUTS_DIR

# %% [markdown]
# ## Load bases

# %%
member_master = pd.read_csv(PROCESSED_DIR / "member_master.csv")
claims_analytical_base = pd.read_csv(PROCESSED_DIR / "claims_analytical_base.csv")
provider_master = pd.read_csv(PROCESSED_DIR / "provider_master.csv")

member_risk_scores = None
risk_scores_path = OUTPUTS_DIR / "member_risk_scores.csv"
if risk_scores_path.exists():
    member_risk_scores = pd.read_csv(risk_scores_path)
    print("member_risk_scores:", member_risk_scores.shape)

print("member_master:", member_master.shape)
print("claims_analytical_base:", claims_analytical_base.shape)
print("provider_master:", provider_master.shape)

# %% [markdown]
# ## Detect claim cost column

# %%
claim_cost_col = None
for candidate in ["claim_amount", "paid_amount", "approved_amount", "amount_paid", "cost_amount", "total_cost"]:
    if candidate in claims_analytical_base.columns:
        claim_cost_col = candidate
        break

print("Detected claim cost column:", claim_cost_col)

# %% [markdown]
# ## Member-level abuse flags

# %%
member_abuse = member_master.copy()

# aggregate claims by member
if "member_id" in claims_analytical_base.columns:
    if claim_cost_col is not None:
        member_claims = (
            claims_analytical_base.groupby("member_id")[claim_cost_col]
            .agg(["count", "mean", "sum", "max"])
            .reset_index()
        )
        member_claims.columns = [
            "member_id",
            "claims_count",
            "claim_cost_mean",
            "claim_cost_sum",
            "claim_cost_max",
        ]
    else:
        member_claims = (
            claims_analytical_base.groupby("member_id")
            .size()
            .reset_index(name="claims_count")
        )

    member_abuse = member_abuse.merge(member_claims, on="member_id", how="left")

for col in ["claims_count", "claim_cost_mean", "claim_cost_sum", "claim_cost_max"]:
    if col in member_abuse.columns:
        member_abuse[col] = member_abuse[col].fillna(0)

if member_risk_scores is not None:
    member_abuse = member_abuse.merge(
        member_risk_scores[["member_id", "predicted_risk_probability", "predicted_risk_segment"]],
        on="member_id",
        how="left"
    )

# %% [markdown]
# ## Member rules

# %%
# claims frequency anomaly
if "claims_count" in member_abuse.columns:
    p95_claims = member_abuse["claims_count"].quantile(0.95)
    member_abuse["flag_extreme_claim_frequency"] = (member_abuse["claims_count"] >= p95_claims).astype(int)
else:
    member_abuse["flag_extreme_claim_frequency"] = 0

# total cost anomaly
if "claim_cost_sum" in member_abuse.columns:
    p95_cost = member_abuse["claim_cost_sum"].quantile(0.95)
    member_abuse["flag_extreme_total_cost"] = (member_abuse["claim_cost_sum"] >= p95_cost).astype(int)
else:
    member_abuse["flag_extreme_total_cost"] = 0

# mismatch between expected low risk and high observed use
if {"predicted_risk_probability", "claims_count"}.issubset(member_abuse.columns):
    low_expected = member_abuse["predicted_risk_probability"] <= member_abuse["predicted_risk_probability"].quantile(0.30)
    high_observed = member_abuse["claims_count"] >= member_abuse["claims_count"].quantile(0.90)
    member_abuse["flag_low_expected_high_use"] = (low_expected & high_observed).astype(int)
else:
    member_abuse["flag_low_expected_high_use"] = 0

# misuse exposure from synthetic member profile
if "high_misuse_exposure_flag" not in member_abuse.columns:
    if "misuse_exposure_propensity" in member_abuse.columns:
        member_abuse["high_misuse_exposure_flag"] = (
            member_abuse["misuse_exposure_propensity"] >= member_abuse["misuse_exposure_propensity"].quantile(0.80)
        ).astype(int)
    else:
        member_abuse["high_misuse_exposure_flag"] = 0

member_rule_cols = [
    "flag_extreme_claim_frequency",
    "flag_extreme_total_cost",
    "flag_low_expected_high_use",
    "high_misuse_exposure_flag",
]

member_abuse["member_abuse_score"] = member_abuse[member_rule_cols].sum(axis=1)

member_abuse["member_abuse_severity"] = pd.cut(
    member_abuse["member_abuse_score"],
    bins=[-1, 0, 1, 2, 10],
    labels=["none", "low", "medium", "high"]
)

# reason text
def build_member_reason(row):
    reasons = []
    if row["flag_extreme_claim_frequency"] == 1:
        reasons.append("extreme_claim_frequency")
    if row["flag_extreme_total_cost"] == 1:
        reasons.append("extreme_total_cost")
    if row["flag_low_expected_high_use"] == 1:
        reasons.append("low_expected_high_use")
    if row["high_misuse_exposure_flag"] == 1:
        reasons.append("high_misuse_exposure")
    return "; ".join(reasons) if reasons else "none"

member_abuse["member_abuse_reason"] = member_abuse.apply(build_member_reason, axis=1)

# %%
member_abuse_flags = member_abuse[[
    c for c in [
        "member_id",
        "age",
        "sex",
        "region",
        "archetype",
        "plan_type",
        "claims_count",
        "claim_cost_sum",
        "misuse_exposure_propensity",
        "predicted_risk_probability",
        "flag_extreme_claim_frequency",
        "flag_extreme_total_cost",
        "flag_low_expected_high_use",
        "high_misuse_exposure_flag",
        "member_abuse_score",
        "member_abuse_severity",
        "member_abuse_reason",
    ] if c in member_abuse.columns
]].copy()

display(
    member_abuse_flags.sort_values("member_abuse_score", ascending=False).head(20)
)

# %% [markdown]
# ## Provider-level fraud / anomaly flags

# %%
provider_fraud = provider_master.copy()

# Convertir posibles columnas numéricas mal tipadas
for col in [
    "base_cost_multiplier",
    "diagnostic_intensity",
    "admission_intensity",
    "fraud_exposure_score",
    "historical_suspicion_flag",
    "observed_vs_expected_claim_ratio",
    "claims_count",
    "claim_cost_mean",
    "claim_cost_sum",
]:
    if col in provider_fraud.columns:
        provider_fraud[col] = pd.to_numeric(provider_fraud[col], errors="coerce")

provider_rule_cols = []

if "claims_count" in provider_fraud.columns:
    provider_fraud["flag_high_claim_volume"] = (
        provider_fraud["claims_count"] >= provider_fraud["claims_count"].quantile(0.95)
    ).astype(int)
    provider_rule_cols.append("flag_high_claim_volume")
else:
    provider_fraud["flag_high_claim_volume"] = 0

if "base_cost_multiplier" in provider_fraud.columns:
    provider_fraud["flag_high_base_cost_multiplier"] = (
        provider_fraud["base_cost_multiplier"] >= provider_fraud["base_cost_multiplier"].quantile(0.90)
    ).astype(int)
    provider_rule_cols.append("flag_high_base_cost_multiplier")
else:
    provider_fraud["flag_high_base_cost_multiplier"] = 0

if "diagnostic_intensity" in provider_fraud.columns:
    provider_fraud["flag_high_diagnostic_intensity"] = (
        provider_fraud["diagnostic_intensity"] >= provider_fraud["diagnostic_intensity"].quantile(0.90)
    ).astype(int)
    provider_rule_cols.append("flag_high_diagnostic_intensity")
else:
    provider_fraud["flag_high_diagnostic_intensity"] = 0

if "admission_intensity" in provider_fraud.columns:
    provider_fraud["flag_high_admission_intensity"] = (
        provider_fraud["admission_intensity"] >= provider_fraud["admission_intensity"].quantile(0.90)
    ).astype(int)
    provider_rule_cols.append("flag_high_admission_intensity")
else:
    provider_fraud["flag_high_admission_intensity"] = 0

if "fraud_exposure_score" in provider_fraud.columns:
    provider_fraud["flag_high_fraud_exposure"] = (
        provider_fraud["fraud_exposure_score"] >= provider_fraud["fraud_exposure_score"].quantile(0.90)
    ).astype(int)
    provider_rule_cols.append("flag_high_fraud_exposure")
else:
    provider_fraud["flag_high_fraud_exposure"] = 0

if "historical_suspicion_flag" in provider_fraud.columns:
    provider_fraud["flag_historical_suspicion"] = provider_fraud["historical_suspicion_flag"].fillna(0).astype(int)
    provider_rule_cols.append("flag_historical_suspicion")
else:
    provider_fraud["flag_historical_suspicion"] = 0

if "observed_vs_expected_claim_ratio" in provider_fraud.columns:
    provider_fraud["flag_high_observed_vs_expected"] = (
        provider_fraud["observed_vs_expected_claim_ratio"] >= provider_fraud["observed_vs_expected_claim_ratio"].quantile(0.90)
    ).astype(int)
    provider_rule_cols.append("flag_high_observed_vs_expected")
else:
    provider_fraud["flag_high_observed_vs_expected"] = 0

provider_fraud["provider_fraud_score"] = provider_fraud[provider_rule_cols].sum(axis=1)

provider_fraud["provider_fraud_severity"] = pd.cut(
    provider_fraud["provider_fraud_score"],
    bins=[-1, 0, 1, 2, 10],
    labels=["none", "low", "medium", "high"]
)

def build_provider_reason(row):
    reasons = []
    for col, label in [
        ("flag_high_claim_volume", "high_claim_volume"),
        ("flag_high_base_cost_multiplier", "high_base_cost_multiplier"),
        ("flag_high_diagnostic_intensity", "high_diagnostic_intensity"),
        ("flag_high_admission_intensity", "high_admission_intensity"),
        ("flag_high_fraud_exposure", "high_fraud_exposure"),
        ("flag_historical_suspicion", "historical_suspicion"),
        ("flag_high_observed_vs_expected", "high_observed_vs_expected"),
    ]:
        if col in row.index and row[col] == 1:
            reasons.append(label)
    return "; ".join(reasons) if reasons else "none"

provider_fraud["provider_fraud_reason"] = provider_fraud.apply(build_provider_reason, axis=1)

provider_fraud_flags = provider_fraud[[
    c for c in [
        "provider_id",
        "provider_name",
        "provider_type",
        "specialty_group",
        "region",
        "provider_archetype",
        "contract_type",
        "base_cost_multiplier",
        "diagnostic_intensity",
        "admission_intensity",
        "average_claim_expected",
        "historical_suspicion_flag",
        "fraud_exposure_score",
        "claims_count",
        "claim_cost_mean",
        "claim_cost_sum",
        "observed_vs_expected_claim_ratio",
        "flag_high_claim_volume",
        "flag_high_base_cost_multiplier",
        "flag_high_diagnostic_intensity",
        "flag_high_admission_intensity",
        "flag_high_fraud_exposure",
        "flag_historical_suspicion",
        "flag_high_observed_vs_expected",
        "provider_fraud_score",
        "provider_fraud_severity",
        "provider_fraud_reason",
    ] if c in provider_fraud.columns
]].copy()

display(
    provider_fraud_flags.sort_values("provider_fraud_score", ascending=False).head(20)
)

# %% [markdown]
# ## Save outputs

# %%
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

member_file = OUTPUTS_DIR / "member_abuse_flags.csv"
provider_file = OUTPUTS_DIR / "provider_fraud_flags.csv"

member_abuse_flags.to_csv(member_file, index=False)
provider_fraud_flags.to_csv(provider_file, index=False)

print("Saved:", member_file)
print("Saved:", provider_file)

# %% [markdown]
# ## Final notes

# %%
notes = [
    "This is the first rule-based fraud / abuse framework of the project.",
    "It is intentionally explainable and business-friendly.",
    "Next step should be pricing adequacy analysis."
]

for i, note in enumerate(notes, 1):
    print(f"{i}. {note}")

PROJECT_ROOT: C:\Users\dolivares\Desktop\novares-securehealth
member_risk_scores: (4000, 17)
member_master: (4000, 63)
claims_analytical_base: (44144, 107)
provider_master: (180, 39)
Detected claim cost column: None


,member_id,age,sex,region,archetype,plan_type,claims_count,misuse_exposure_propensity,predicted_risk_probability,flag_extreme_claim_frequency,flag_extreme_total_cost,flag_low_expected_high_use,high_misuse_exposure_flag,member_abuse_score,member_abuse_severity,member_abuse_reason
3139,MBR003140,56,F,Paraguarí,Hyper-Utilizer / Misuse Risk,Managed Review,46.0,0.640,0.987620,1,0,0,1,2,medium,extreme_claim_frequency; high_misuse_exposure
2897,MBR002898,24,M,Central,Hyper-Utilizer / Misuse Risk,Managed Review,50.0,0.627,0.911709,1,0,0,1,2,medium,extreme_claim_frequency; high_misuse_exposure
27,MBR000028,61,F,Central,Hyper-Utilizer / Misuse Risk,Managed Review,56.0,0.582,0.997864,1,0,0,1,2,medium,extreme_claim_frequency; high_misuse_exposure
3244,MBR003245,64,F,San Pedro,Hyper-Utilizer / Misuse Risk,Managed Review,44.0,0.563,0.995399,1,0,0,1,2,medium,extreme_claim_frequency; high_misuse_exposure
391,MBR000392,29,M,Central,Hyper-Utilizer / Misuse Risk,Managed Review,52.0,0.640,1.000000,1,0,0,1,2,medium,extreme_claim_frequency; high_misuse_exposure
2712,MBR002713,62,M,Asunción,Hyper-Utilizer / Misuse Risk,Managed Review,56.0,0.887,0.999255,1,0,0,1,2,medium,extreme_claim_frequency; high_misuse_exposure
3575,MBR003576,49,F,Asunción,Hyper-Utilizer / Misuse Risk,Managed Review,56.0,0.609,0.996899,1,0,0,1,2,medium,extreme_claim_frequency; high_misuse_exposure
3220,MBR003221,31,F,Itapúa,Hyper-Utilizer / Misuse Risk,Managed Review,56.0,0.595,0.996253,1,0,0,1,2,medium,extreme_claim_frequency; high_misuse_exposure
3222,MBR003223,67,M,Central,Chronic Managed,Chronic Care,37.0,0.126,0.990625,1,0,0,1,2,medium,extreme_claim_frequency; high_misuse_exposure
3230,MBR003231,39,F,Central,Hyper-Utilizer / Misuse Risk,Managed Review,44.0,0.758,0.993734,1,0,0,1,2,medium,extreme_claim_frequency; high_misuse_exposure


,provider_id,provider_name,provider_type,specialty_group,region,provider_archetype,contract_type,base_cost_multiplier,diagnostic_intensity,admission_intensity,...,flag_high_claim_volume,flag_high_base_cost_multiplier,flag_high_diagnostic_intensity,flag_high_admission_intensity,flag_high_fraud_exposure,flag_historical_suspicion,flag_high_observed_vs_expected,provider_fraud_score,provider_fraud_severity,provider_fraud_reason
3,PRV00004,Provider 4,Lab,Laboratory,Alto Paraná,Suspicious,Standard,1.939815,NaN,NaN,...,0,1,0,0,1,1,0,3,high,high_base_cost_multiplier; high_fraud_exposure...
6,PRV00007,Provider 7,Hospital,General Medicine,Paraguarí,Suspicious,Preferred,1.840071,NaN,NaN,...,0,1,0,0,1,1,0,3,high,high_base_cost_multiplier; high_fraud_exposure...
69,PRV00070,Provider 70,Specialist Center,Pulmonology,San Pedro,Suspicious,Preferred,2.314093,NaN,NaN,...,0,1,0,0,1,1,0,3,high,high_base_cost_multiplier; high_fraud_exposure...
72,PRV00073,Provider 73,Hospital,Orthopedics,Itapúa,Suspicious,Standard,1.715096,NaN,NaN,...,0,1,0,0,1,1,0,3,high,high_base_cost_multiplier; high_fraud_exposure...
134,PRV00135,Provider 135,Specialist Center,Pulmonology,Central,Suspicious,Standard,1.672246,NaN,NaN,...,0,1,0,0,1,1,0,3,high,high_base_cost_multiplier; high_fraud_exposure...
156,PRV00157,Provider 157,Imaging,Radiology,Alto Paraná,Suspicious,External,1.959505,NaN,NaN,...,0,1,0,0,1,1,0,3,high,high_base_cost_multiplier; high_fraud_exposure...
171,PRV00172,Provider 172,Hospital,Orthopedics,Alto Paraná,Suspicious,External,2.055429,NaN,NaN,...,0,1,0,0,1,1,0,3,high,high_base_cost_multiplier; high_fraud_exposure...
112,PRV00113,Provider 113,Specialist Center,Neurology,Asunción,Suspicious,Standard,1.796325,NaN,NaN,...,0,1,0,0,1,1,0,3,high,high_base_cost_multiplier; high_fraud_exposure...
84,PRV00085,Provider 85,Clinic,Cardiology,Central,Suspicious,Standard,1.847824,NaN,NaN,...,0,1,0,0,1,1,0,3,high,high_base_cost_multiplier; high_fraud_exposure...
90,PRV00091,Provider 91,Hospital,General Medicine,Alto Paraná,Suspicious,Standard,1.723820,NaN,NaN,...,0,1,0,0,1,1,0,3,high,high_base_cost_multiplier; high_fraud_exposure...


Saved: C:\Users\dolivares\Desktop\novares-securehealth\data\outputs\member_abuse_flags.csv
Saved: C:\Users\dolivares\Desktop\novares-securehealth\data\outputs\provider_fraud_flags.csv
1. This is the first rule-based fraud / abuse framework of the project.
2. It is intentionally explainable and business-friendly.
3. Next step should be pricing adequacy analysis.
